In [1]:
from pyspark.sql import SparkSession
# Initialize Spark Session
# Initialize Spark Session with dynamic allocation
spark = SparkSession.builder \
.appName("ReduceByKey_300mb") \
.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/06/29 22:15:00 WARN Utils: Your hostname, Onkars-MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.140 instead (on interface en0)
25/06/29 22:15:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/29 22:15:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
!hadoop fs -ls -h /tmp

zsh:1: command not found: hadoop


In [3]:
!hadoop fs -head /tmp/customers.csv

zsh:1: command not found: hadoop


In [4]:
hdfs_path = "../data/ecommerce_data/1MB/customers.csv"

rdd = spark.sparkContext.textFile(hdfs_path)

In [5]:
header = rdd.first()

In [6]:
rdd_no_header = rdd.filter(lambda row:row !=header).map(lambda row:row.split(','))

In [7]:
rdd_no_header.first()

['0', 'Customer_0', 'Pune', 'Maharashtra', 'India', '2023-06-29', 'False']

In [8]:
key_value_rdd = rdd_no_header.map(lambda row:(row[2],1))

In [9]:
reduced_rdd = key_value_rdd.reduceByKey(lambda x,y : x+y)

In [10]:
reduced_rdd.collect()

[('Pune', 2243),
 ('Hyderabad', 2242),
 ('Mumbai', 2142),
 ('Delhi', 2200),
 ('Bangalore', 2211),
 ('Ahmedabad', 2198),
 ('Chennai', 2194),
 ('Kolkata', 2223)]

In [11]:
spark.stop()

In [12]:
from pyspark.sql import SparkSession
# Initialize Spark Session
# Initialize Spark Session with dynamic allocation
spark = SparkSession.builder \
.appName("GroupByKey_300mb_2") \
.getOrCreate()

In [13]:
hdfs_path = "../data/ecommerce_data/1MB/customers.csv"

rdd = spark.sparkContext.textFile(hdfs_path)

In [14]:
header = rdd.first()
rdd_no_header = rdd.filter(lambda row:row !=header).map(lambda row:row.split(','))

In [ ]:
key_value_rdd = rdd_no_header.map(lambda row:(row[2],1))

In [16]:
grouped_rdd = key_value_rdd.groupByKey()

In [17]:
result = grouped_rdd.map(lambda x : (x[0],len(x[1])))

In [18]:
result.collect()

[('Pune', 2243),
 ('Hyderabad', 2242),
 ('Mumbai', 2142),
 ('Delhi', 2200),
 ('Bangalore', 2211),
 ('Ahmedabad', 2198),
 ('Chennai', 2194),
 ('Kolkata', 2223)]

In [ ]:
[('Delhi', 661025),
 ('Pune', 660737),
 ('Kolkata', 660174),
 ('Chennai', 660249),
 ('Bangalore', 661013),
 ('Mumbai', 661241),
 ('Hyderabad', 662281),
 ('Ahmedabad', 660218)]

In [19]:
spark.stop()

In [20]:
spark = SparkSession.builder \
.appName ("Repartition_vs_Coalesce") \
.config("spark.executor.memory", "2g") \
.config("spark.executor.cores", "2") \
.getOrCreate()

In [21]:
hdfs_path = "../data/ecommerce_data/1MB/customers.csv"

rdd = spark.sparkContext.textFile(hdfs_path)

In [22]:
rdd.getNumPartitions()

2

In [23]:
repartitioned_rdd = rdd.repartition(4)

In [24]:
repartitioned_rdd.getNumPartitions()

4

In [25]:
reparitioned_less_rdd = rdd.repartition(2)

In [26]:
reparitioned_less_rdd.getNumPartitions()

2

In [27]:
reparitioned_more_rdd = rdd.repartition(200)

In [28]:
reparitioned_more_rdd.getNumPartitions()

200

In [29]:
reparitioned_more_rdd.collect()

['1709,Customer_1709,Pune,Delhi,India,2023-05-12,True',
 '1710,Customer_1710,Delhi,Tamil Nadu,India,2023-02-16,False',
 '1711,Customer_1711,Kolkata,Maharashtra,India,2023-12-06,False',
 '1712,Customer_1712,Pune,Tamil Nadu,India,2023-03-30,False',
 '1713,Customer_1713,Hyderabad,Karnataka,India,2023-02-14,True',
 '1714,Customer_1714,Bangalore,West Bengal,India,2023-11-17,True',
 '1715,Customer_1715,Bangalore,Telangana,India,2023-07-13,False',
 '1716,Customer_1716,Chennai,Delhi,India,2023-01-06,False',
 '1717,Customer_1717,Kolkata,Telangana,India,2023-09-20,True',
 '1718,Customer_1718,Kolkata,West Bengal,India,2023-07-15,False',
 '3709,Customer_3709,Ahmedabad,Karnataka,India,2023-09-07,False',
 '3710,Customer_3710,Pune,Telangana,India,2023-08-31,True',
 '3711,Customer_3711,Kolkata,Delhi,India,2023-01-02,True',
 '3712,Customer_3712,Hyderabad,Telangana,India,2023-05-23,False',
 '3713,Customer_3713,Kolkata,Delhi,India,2023-06-18,True',
 '3714,Customer_3714,Pune,Delhi,India,2023-03-04,False',

In [30]:
coalesce_rdd = reparitioned_more_rdd.coalesce(4)

In [31]:
coalesce_rdd.getNumPartitions()

4

In [32]:
coalesce_rdd.count()

17654

In [33]:
spark.stop()